#### Import required libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

#### Reading our dataset

In [2]:
df = pd.read_csv("ai-layoffs-register.csv")
df.head()

,Company,Date,Sector,Country,Functions,Roles counted,Roles reported (not counted),Why not counted,% of workforce,Evidence tier,Attribution,Execution,Counts in US index,Claim,Source,Source URL,Event ID,Kind,What the employer said,Job families
0,Barclays,2026-10-22,Banking,United Kingdom,"Support functions, including software develope...",5000.0,NaN,NaN,about 5% of employees (Barclays' own figure; t...,Tier 1,mixed,partial,no,Barclays' Q3 2026 results presentation (22 Oct...,Barclays: Q3 2026 Results Presentation,https://home.barclays/content/dam/home-barclay...,barclays-2026-10,cut,Cited AI among other reasons,Corporate functions; Software engineering and IT
1,National Australia Bank,2026-10-20,Banking,Australia,Corporate functions (per NAB); Senior manageme...,NaN,2500.0,The company said the cut was not caused by AI;...,up to about 6% of its internal workforce (NAB),Tier 1,blamed,announced,no,"The Nightly (October 20, 2026): ""Kyle Rodda, a...",NAB (ASX announcement): NAB simplifying operat...,https://www.nab.com.au/news/nab-updates/nab-si...,national-australia-bank-2026-10,cut,Denied the AI link others made,Corporate functions; Management layers; Operat...
2,Vermont Information Processing,2026-10-20,Beverage industry software,United States,Not specified,NaN,NaN,NaN,NaN,Tier 2,blamed,executed,no,"Seven Days (October 20, 2026): ""The move appea...",Seven Days: A Colchester Tech Firm Has Been Le...,https://www.sevendaysvt.com/news/a-colchester-...,vermont-information-processing-2026-10,cut,Named no AI reason; others linked it to AI,Not specified
3,HDFC Bank,2026-10-17,Banking,India,No roles cut (a projection: fewer people in ba...,NaN,NaN,NaN,NaN,Tier 1,mixed,announced,no,"Sashidhar Jagdishan, outgoing managing directo...",Investing.com: Earnings call transcript: HDFC ...,https://ng.investing.com/news/stock-market-new...,hdfc-bank-2026-10,workforce statement,Spoke about AI and headcount; no cut announced,Not specified
4,Lookiero,2026-10-16,Online personal styling (fashion e-commerce),Spain,Not specified (Gananzia's internal sources pla...,NaN,50.0,"The number was reported by the press, not stat...",about 10% of the group's staff (the company's ...,Tier 2,mixed,announced,no,"Lookiero Outfittery Group, in its official res...",Gananzia: Lookiero prepara un ERE pese a recib...,https://gananzia.com/lookiero-prepara-un-ere-p...,lookiero-2026-10,cut,Cited AI among other reasons,Not specified



| Column | Meaning |
| --- | --- |
| `Company` | Employer name as it appears in the source. |
| `Date` | Event date (ISO `YYYY-MM-DD`): the announcement, filing, or effective date. |
| `Sector` | The employer's sector. |
| `Country` | The employer's home country (headquarters or domicile), as free text; for a few events it names where the roles were instead, or notes that the employer is global. |
| `Functions` | The roles cut, where a source says which, as entries separated by "; " (no entry contains a semicolon). An entry that comes from reporting names the outlet ("per ..."); when no source says which roles went, the entry begins "Not specified". |
| `Roles counted` | Headcount that meets the register's standard for an AI-attributed cut and enters the worldwide totals. Blank when no qualifying figure exists. |
| `Roles reported (not counted)` | Headcount reported for the event that the register deliberately does not count. Kept separate so counted and reported facts are never conflated. |
| `Why not counted` | The grounds for setting that reported figure aside, in plain words: the company said the cut was not caused by AI, or it has not named AI as the cause; or the number is not a figure the company stated as a cut (reported by the press or given by a union rather than stated by the company, an upper limit, a net change in headcount, roles shed through attrition, or a count that includes contractors or roles moved to other teams or locations). |
| `% of workforce` | Share of the workforce affected, where the source expresses it. |
| `Evidence tier` | Strength of the evidence for the AI attribution. Tier 1 = primary filing or first-party company statement; Tier 2 = named, credible reporting; Tier 3 = weaker or single-source signal. |
| `Attribution` | How strongly the cut is tied to AI. `explicit` = the employer itself names AI as the cause; `mixed` = AI is one stated factor among others (efficiency, restructuring); `blamed` = a credible source names AI but the employer does not, so the row is never counted. |
| `Execution` | Status: `announced`, `executed`, `partial`, `reversed` or `unknown`. |
| `Counts in US index` | `yes` only when a counted figure covers US-based roles, so it enters the US index sum. `no` for every other row, including US-based rows whose figure is not counted. |
| `Claim` | A short, sourced description of exactly what was disclosed and how it is classified. |
| `Source` | The name of the source (for example, a specific Form 8-K or a named news report). |
| `Source URL` | A resolvable link to that source, free to open. |
| `Event ID` | The register's stable identifier for the event (for example `cisco-2024-08`), the same id the JSON API uses. Added in 1.35. |
| `Kind` | `cut` for a layoff or workforce reduction; `workforce statement` for an employer's public statement about AI and the size of its workforce made when no discrete cut was announced (a forecast, or a headcount said to have fallen through attrition). A statement never carries a counted figure; leave these rows out of any count of cuts. Added in 1.35. |
| `What the employer said` | The employer's own position on AI for this event, in one of seven fixed labels the site also shows: "Declared the cut AI-related", "Cited AI among other reasons", "Cited AI, but also said AI was not a reason", "Credited AI; reduced through attrition, not layoffs", "Denied the AI link others made", "Named no AI reason; others linked it to AI", "Spoke about AI and headcount; no cut announced". Added in 1.35. |
| `Job families` | The kinds of work cut, coded by the register from the roles its sources name, separated by "; ": Customer service and support, Operations and back office, Software engineering and IT, Corporate functions, Sales and marketing, Management layers, Product and design, Writing, editorial and translation, Content moderation, Frontline and manual work, Consulting and professional services, or Not specified when no source says which roles went. Added in 1.35. |


#### Exploring and Cleaning each and every column

In [3]:
df.head()

,Company,Date,Sector,Country,Functions,Roles counted,Roles reported (not counted),Why not counted,% of workforce,Evidence tier,Attribution,Execution,Counts in US index,Claim,Source,Source URL,Event ID,Kind,What the employer said,Job families
0,Barclays,2026-10-22,Banking,United Kingdom,"Support functions, including software develope...",5000.0,NaN,NaN,about 5% of employees (Barclays' own figure; t...,Tier 1,mixed,partial,no,Barclays' Q3 2026 results presentation (22 Oct...,Barclays: Q3 2026 Results Presentation,https://home.barclays/content/dam/home-barclay...,barclays-2026-10,cut,Cited AI among other reasons,Corporate functions; Software engineering and IT
1,National Australia Bank,2026-10-20,Banking,Australia,Corporate functions (per NAB); Senior manageme...,NaN,2500.0,The company said the cut was not caused by AI;...,up to about 6% of its internal workforce (NAB),Tier 1,blamed,announced,no,"The Nightly (October 20, 2026): ""Kyle Rodda, a...",NAB (ASX announcement): NAB simplifying operat...,https://www.nab.com.au/news/nab-updates/nab-si...,national-australia-bank-2026-10,cut,Denied the AI link others made,Corporate functions; Management layers; Operat...
2,Vermont Information Processing,2026-10-20,Beverage industry software,United States,Not specified,NaN,NaN,NaN,NaN,Tier 2,blamed,executed,no,"Seven Days (October 20, 2026): ""The move appea...",Seven Days: A Colchester Tech Firm Has Been Le...,https://www.sevendaysvt.com/news/a-colchester-...,vermont-information-processing-2026-10,cut,Named no AI reason; others linked it to AI,Not specified
3,HDFC Bank,2026-10-17,Banking,India,No roles cut (a projection: fewer people in ba...,NaN,NaN,NaN,NaN,Tier 1,mixed,announced,no,"Sashidhar Jagdishan, outgoing managing directo...",Investing.com: Earnings call transcript: HDFC ...,https://ng.investing.com/news/stock-market-new...,hdfc-bank-2026-10,workforce statement,Spoke about AI and headcount; no cut announced,Not specified
4,Lookiero,2026-10-16,Online personal styling (fashion e-commerce),Spain,Not specified (Gananzia's internal sources pla...,NaN,50.0,"The number was reported by the press, not stat...",about 10% of the group's staff (the company's ...,Tier 2,mixed,announced,no,"Lookiero Outfittery Group, in its official res...",Gananzia: Lookiero prepara un ERE pese a recib...,https://gananzia.com/lookiero-prepara-un-ere-p...,lookiero-2026-10,cut,Cited AI among other reasons,Not specified


**Company Column**

In [4]:
list_of_companies_impacted_by_ai_layoffs = df["Company"].unique().tolist()


print("List of Companies Impacted by AI Layoffs")
print()
print(list_of_companies_impacted_by_ai_layoffs)
print()
print()
print("Number of such companies : ", len(list_of_companies_impacted_by_ai_layoffs))

List of Companies Impacted by AI Layoffs

['Barclays', 'National Australia Bank', 'Vermont Information Processing', 'HDFC Bank', 'Lookiero', 'Anthropic', 'Bank of America', 'GSK', 'KLM', 'Kroger', 'One Zero', 'HoneyBook', 'HSBC', 'DNB', 'FICO (Fair Isaac)', 'HubSpot', 'CWT Italia (Carlson Wagonlit Travel)', 'Tamedia', 'F-Secure', 'BMW', 'Cariad (Volkswagen)', 'Future plc', 'Disney', 'Thinkific', 'Sunrise', 'Worldia', 'Reach plc', 'Ouest-France', 'Wells Fargo', 'Oracle', 'Konecta', 'Commerce.com (BigCommerce)', 'IONOS', 'Booksy', 'McClatchy', 'Axis Bank', 'The Bancorp', 'Uber', 'Zscaler, Inc.', 'WPP', 'Zomato', 'Cubic3', 'PagerDuty, Inc.', 'Apple', 'Pentera', 'Rapid7', 'VideoAmp', 'LegalZoom', 'TikTok', 'Sparkgeo', 'E.W. Scripps', 'Latch, Inc. (DOOR)', 'Chime', 'Visa Inc.', 'Commonwealth Bank', 'Papaya Gaming', 'Patreon', 'monday.com', 'Amazon', 'CDW Corporation', 'Sprout Social, Inc.', 'Freight Technologies (Fr8Tech)', 'JPMorgan Chase', 'Thomson Reuters Corporation', 'GoKwik', 'Bragg G

Checking for null values

In [5]:
df["Company"].isna().sum()

np.int64(0)

**Converting the date column to pandas datetime format**

In [8]:
df["Date"] = pd.to_datetime(df["Date"])

In [6]:
df.head()

,Company,Date,Sector,Country,Functions,Roles counted,Roles reported (not counted),Why not counted,% of workforce,Evidence tier,Attribution,Execution,Counts in US index,Claim,Source,Source URL,Event ID,Kind,What the employer said,Job families
0,Barclays,2026-10-22,Banking,United Kingdom,"Support functions, including software develope...",5000.0,NaN,NaN,about 5% of employees (Barclays' own figure; t...,Tier 1,mixed,partial,no,Barclays' Q3 2026 results presentation (22 Oct...,Barclays: Q3 2026 Results Presentation,https://home.barclays/content/dam/home-barclay...,barclays-2026-10,cut,Cited AI among other reasons,Corporate functions; Software engineering and IT
1,National Australia Bank,2026-10-20,Banking,Australia,Corporate functions (per NAB); Senior manageme...,NaN,2500.0,The company said the cut was not caused by AI;...,up to about 6% of its internal workforce (NAB),Tier 1,blamed,announced,no,"The Nightly (October 20, 2026): ""Kyle Rodda, a...",NAB (ASX announcement): NAB simplifying operat...,https://www.nab.com.au/news/nab-updates/nab-si...,national-australia-bank-2026-10,cut,Denied the AI link others made,Corporate functions; Management layers; Operat...
2,Vermont Information Processing,2026-10-20,Beverage industry software,United States,Not specified,NaN,NaN,NaN,NaN,Tier 2,blamed,executed,no,"Seven Days (October 20, 2026): ""The move appea...",Seven Days: A Colchester Tech Firm Has Been Le...,https://www.sevendaysvt.com/news/a-colchester-...,vermont-information-processing-2026-10,cut,Named no AI reason; others linked it to AI,Not specified
3,HDFC Bank,2026-10-17,Banking,India,No roles cut (a projection: fewer people in ba...,NaN,NaN,NaN,NaN,Tier 1,mixed,announced,no,"Sashidhar Jagdishan, outgoing managing directo...",Investing.com: Earnings call transcript: HDFC ...,https://ng.investing.com/news/stock-market-new...,hdfc-bank-2026-10,workforce statement,Spoke about AI and headcount; no cut announced,Not specified
4,Lookiero,2026-10-16,Online personal styling (fashion e-commerce),Spain,Not specified (Gananzia's internal sources pla...,NaN,50.0,"The number was reported by the press, not stat...",about 10% of the group's staff (the company's ...,Tier 2,mixed,announced,no,"Lookiero Outfittery Group, in its official res...",Gananzia: Lookiero prepara un ERE pese a recib...,https://gananzia.com/lookiero-prepara-un-ere-p...,lookiero-2026-10,cut,Cited AI among other reasons,Not specified


In [8]:
st = SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [20]:
list_of_sectors = df["Sector"].unique().tolist()
encoded_sentences = pd.DataFrame(st.encode(list_of_sectors),index=list_of_sectors)
similarity_matrix = pd.DataFrame(cosine_similarity(encoded_sentences), index = list_of_sectors,columns=list_of_sectors).reset_index()
similarity_matrix_1 = similarity_matrix.copy()

modified_list_of_sectors = list_of_sectors.copy()
sectors_to_be_ignored = []
new_list_of_sectors = []
industry_groups = {}

#i = 0
ctr =0
while True:
    try:
        single_sector = similarity_matrix.iloc[:,[0,1]].sort_values(by=similarity_matrix.columns[1],ascending=False).reset_index(drop=True)
    except:
        break
    pl = single_sector[single_sector.iloc[:,1]>0.40]
    cols_to_be_excluded = pl["index"].values
    new_list_of_sectors.append(cols_to_be_excluded[0])
    industry_groups[cols_to_be_excluded[0]] = cols_to_be_excluded[1:].tolist()
    similarity_matrix = similarity_matrix[~similarity_matrix["index"].isin(cols_to_be_excluded)].drop(columns=cols_to_be_excluded).reset_index(drop=True)
    ctr += 1
new_list_of_sectors.sort()
new_list_of_sectors

['AI transcription and captioning',
 'Advertising measurement software',
 'Airlines / aviation group',
 'Appointment booking software (beauty and wellness)',
 'Artificial intelligence (AI models and software)',
 'Automotive',
 'B2B sales data software',
 'Banking',
 'Beverage industry software',
 'Blockchain and AI infrastructure startup (crypto)',
 'Business travel management',
 'Chemicals / Materials',
 'Contact centres and business process outsourcing (CRM/BPO)',
 'Creative technology (stock media and AI video tools)',
 'Creator membership platform',
 'Crypto exchange',
 'Cybersecurity software',
 'Fintech / Payments (AP/AR automation)',
 'Freight logistics software and brokerage',
 'Geospatial data engineering services',
 'Grocery retail',
 'Health tech (Alphabet)',
 'Healthcare / Revenue cycle',
 'IT solutions / Value-added reseller',
 'Insurance',
 'Legal technology',
 'Live events ticketing',
 'Media (newspapers)',
 'Online course software',
 'Online personal styling (fashion e-

In [21]:
industry_groups

{'Banking': ['Banking / financial services',
  'Financial services / Banking',
  'Banking (wealth management)',
  'Banking (fintech partner bank)',
  'Digital banking',
  'Financial services / Digital banking',
  'Fintech (neobanking)',
  'Fintech / consumer banking app',
  'Payments / Fintech',
  'Fintech (payments)',
  'Fintech / Payments',
  'Fintech',
  'Crypto / Fintech',
  'Crypto / fintech',
  'Fintech / customer service',
  'E-commerce / fintech',
  'Fintech (digital payments)',
  'Fintech / digital payments',
  'Financial education (edtech)',
  'Cryptocurrency exchange (fintech)',
  'Fintech (custody for investment advisers)',
  'Fintech / payments (one-click checkout)',
  'Insurance brokerage and fintech',
  'Insurance Brokerage / Fintech',
  'Telecom',
  'Telecommunications',
  'Airline'],
 'Beverage industry software': ['Software',
  'Design software',
  'Technology services and software',
  'Software (CRM and marketing platform)',
  'E-commerce marketing software',
  'Soft

In [16]:
industry_reverse_groups = {}
for i in industry_groups:
    if len(industry_groups[i])>0:
        for j in industry_groups[i]:
            industry_reverse_groups[j] = i
    else:
        industry_reverse_groups[i] = i

for i in industry_groups:
    industry_reverse_groups[i] = i

df_1 = df.copy()
df_1["Sector"] = df_1["Sector"].map(industry_reverse_groups)
df_1

,Company,Date,Sector,Country,Functions,Roles counted,Roles reported (not counted),Why not counted,% of workforce,Evidence tier,Attribution,Execution,Counts in US index,Claim,Source,Source URL,Event ID,Kind,What the employer said,Job families
0,Barclays,2026-10-22,"Financial Services, Banking & Fintech",United Kingdom,"Support functions, including software develope...",5000.0,NaN,NaN,about 5% of employees (Barclays' own figure; t...,Tier 1,mixed,partial,no,Barclays' Q3 2026 results presentation (22 Oct...,Barclays: Q3 2026 Results Presentation,https://home.barclays/content/dam/home-barclay...,barclays-2026-10,cut,Cited AI among other reasons,Corporate functions; Software engineering and IT
1,National Australia Bank,2026-10-20,"Financial Services, Banking & Fintech",Australia,Corporate functions (per NAB); Senior manageme...,NaN,2500.0,The company said the cut was not caused by AI;...,up to about 6% of its internal workforce (NAB),Tier 1,blamed,announced,no,"The Nightly (October 20, 2026): ""Kyle Rodda, a...",NAB (ASX announcement): NAB simplifying operat...,https://www.nab.com.au/news/nab-updates/nab-si...,national-australia-bank-2026-10,cut,Denied the AI link others made,Corporate functions; Management layers; Operat...
2,Vermont Information Processing,2026-10-20,"Retail, Consumer Goods & E-commerce",United States,Not specified,NaN,NaN,NaN,NaN,Tier 2,blamed,executed,no,"Seven Days (October 20, 2026): ""The move appea...",Seven Days: A Colchester Tech Firm Has Been Le...,https://www.sevendaysvt.com/news/a-colchester-...,vermont-information-processing-2026-10,cut,Named no AI reason; others linked it to AI,Not specified
3,HDFC Bank,2026-10-17,"Financial Services, Banking & Fintech",India,No roles cut (a projection: fewer people in ba...,NaN,NaN,NaN,NaN,Tier 1,mixed,announced,no,"Sashidhar Jagdishan, outgoing managing directo...",Investing.com: Earnings call transcript: HDFC ...,https://ng.investing.com/news/stock-market-new...,hdfc-bank-2026-10,workforce statement,Spoke about AI and headcount; no cut announced,Not specified
4,Lookiero,2026-10-16,"Retail, Consumer Goods & E-commerce",Spain,Not specified (Gananzia's internal sources pla...,NaN,50.0,"The number was reported by the press, not stat...",about 10% of the group's staff (the company's ...,Tier 2,mixed,announced,no,"Lookiero Outfittery Group, in its official res...",Gananzia: Lookiero prepara un ERE pese a recib...,https://gananzia.com/lookiero-prepara-un-ere-p...,lookiero-2026-10,cut,Cited AI among other reasons,Not specified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,BT Group,2023-05-18,NaN,United Kingdom,"Customer service (about 10,000 roles through d...",NaN,10000.0,"The number includes contractors, not only empl...","of 55,000 total cuts",Tier 1,mixed,announced,no,"BT said it would cut up to 55,000 jobs by 2030...",BT Group Newsroom (FY2023 results),https://newsroom.bt.com/results-for-the-full-y...,bt-group-2023-05,cut,Cited AI among other reasons,Customer service and support
316,IBM,2023-05-01,NaN,United States,Back-office; HR,NaN,7800.0,"The number was reported by the press, not stat...",~30% of back-office roles (projected),Tier 2,mixed,announced,no,CEO Arvind Krishna said he could 'easily see 3...,Bloomberg via The National (Krishna interview),https://www.thenationalnews.com/business/techn...,ibm-2023-2025,workforce statement,Spoke about AI and headcount; no cut announced,Operations and back office; Corporate functions
317,Dropbox,2023-04-27,NaN,United States,Research and development; Sales and marketing;...,500.0,NaN,NaN,about 16%,Tier 1,mixed,executed,no,"Dropbox CEO Drew Houston, letter to employees ...",Dropbox Blog: A message from our CEO (Drew Hou...,https://blog.dropbox.com/topics/company/a-mess...,dropbox-2023-04,cut,Cited AI among other reasons,Software engineering and IT; Sales and marketi...
318,Turnitin,2023-04-18,NaN,United States,No roles cut (a forecast that within 18 months...,NaN,NaN,NaN,NaN,Tier 1,expl

In [24]:
len(df["Sector"].unique().tolist())

228

In [12]:
industry_groups = {
    "Artificial Intelligence, Data & Analytics": [
        "Artificial intelligence (AI models and software)",
        "Analytics software and credit scoring",
        "Audience measurement",
        "Geospatial data engineering services"
    ],

    "Enterprise Software, SaaS & Developer Tools": [
        "Dev Tools",
        "E-commerce enablement / SaaS",
        "HR software",
        "IT solutions / Value-added reseller",
        "Software (Apple device management)",
        "Software / Search & observability",
        "Technology (enterprise software/cloud)",
        "Website-building software (WordPress)"
    ],

    "Financial Services, Banking & Fintech": [
        "Banking",
        "Fintech / Payments (AP/AR automation)",
        "Insurance",
        "Crypto exchange",
        "digital asset custody and infrastructure",
        "Blockchain and AI infrastructure startup (crypto)"
    ],

    "Healthcare, Pharmaceuticals & Life Sciences": [
        "AI software for drug discovery",
        "Health tech (Alphabet)",
        "Healthcare / Revenue cycle",
        "Pharmaceuticals and vaccines"
    ],

    "Media, Entertainment & Creative Technology": [
        "Media (newspapers)",
        "AI transcription and captioning",
        "Creative technology (stock media and AI video tools)",
        "Music licensing and creative software",
        "Video game services (localisation, testing, audio)",
        "mobile games (skill-based competitions)",
        "Live events ticketing",
        "Social media (US joint venture; office including content moderation)",
        "Creator membership platform"
    ],

    "Advertising, Marketing & Communications": [
        "Advertising and marketing services",
        "Business messaging software (CPaaS)",
        "Contact centres and business process outsourcing (CRM/BPO)",
        "gambling affiliate marketing and sports data"
    ],

    "Retail, Consumer Goods & E-commerce": [
        "Grocery retail",
        "Consumer Goods / Tobacco",
        "Online personal styling (fashion e-commerce)",
        "Beverage industry software"
    ],

    "Transportation, Mobility & Logistics": [
        "Airline",
        "Automotive",
        "Freight logistics software and brokerage",
        "Business travel management",
        "Technology / Ridesharing"
    ],

    "Education & Professional Development": [
        "Online course software",
        "tech career upskilling (edtech)",
        "online freelance-work marketplace"
    ],

    "Cybersecurity & Telecommunications": [
        "Cybersecurity software",
        "Telecommunications"
    ],

    "Industrial, Manufacturing & Property Technology": [
        "Chemicals / Materials",
        "Smart building / Property technology software",
        "Technology (VR, metaverse and wearables hardware)"
    ],

    "Legal, Lifestyle & Consumer Services": [
        "Appointment booking software (beauty and wellness)",
        "Online legal services"
    ]
}

In [492]:
industry_reverse_groups = {}
for i in industry_groups:
    for j in industry_groups[i]:
        industry_reverse_groups[j] = i

In [493]:
df_1 = df.copy()
df_1["Sector"] = df_1["Sector"].map(industry_reverse_groups)
df_1

,Company,Date,Sector,Country,Functions,Roles counted,Roles reported (not counted),Why not counted,% of workforce,Evidence tier,Attribution,Execution,Counts in US index,Claim,Source,Source URL,Event ID,Kind,What the employer said,Job families
0,Barclays,2026-10-22,"Financial Services, Banking & Fintech",United Kingdom,"Support functions, including software develope...",5000.0,NaN,NaN,about 5% of employees (Barclays' own figure; t...,Tier 1,mixed,partial,no,Barclays' Q3 2026 results presentation (22 Oct...,Barclays: Q3 2026 Results Presentation,https://home.barclays/content/dam/home-barclay...,barclays-2026-10,cut,Cited AI among other reasons,Corporate functions; Software engineering and IT
1,National Australia Bank,2026-10-20,"Financial Services, Banking & Fintech",Australia,Corporate functions (per NAB); Senior manageme...,NaN,2500.0,The company said the cut was not caused by AI;...,up to about 6% of its internal workforce (NAB),Tier 1,blamed,announced,no,"The Nightly (October 20, 2026): ""Kyle Rodda, a...",NAB (ASX announcement): NAB simplifying operat...,https://www.nab.com.au/news/nab-updates/nab-si...,national-australia-bank-2026-10,cut,Denied the AI link others made,Corporate functions; Management layers; Operat...
2,Vermont Information Processing,2026-10-20,"Retail, Consumer Goods & E-commerce",United States,Not specified,NaN,NaN,NaN,NaN,Tier 2,blamed,executed,no,"Seven Days (October 20, 2026): ""The move appea...",Seven Days: A Colchester Tech Firm Has Been Le...,https://www.sevendaysvt.com/news/a-colchester-...,vermont-information-processing-2026-10,cut,Named no AI reason; others linked it to AI,Not specified
3,HDFC Bank,2026-10-17,"Financial Services, Banking & Fintech",India,No roles cut (a projection: fewer people in ba...,NaN,NaN,NaN,NaN,Tier 1,mixed,announced,no,"Sashidhar Jagdishan, outgoing managing directo...",Investing.com: Earnings call transcript: HDFC ...,https://ng.investing.com/news/stock-market-new...,hdfc-bank-2026-10,workforce statement,Spoke about AI and headcount; no cut announced,Not specified
4,Lookiero,2026-10-16,"Retail, Consumer Goods & E-commerce",Spain,Not specified (Gananzia's internal sources pla...,NaN,50.0,"The number was reported by the press, not stat...",about 10% of the group's staff (the company's ...,Tier 2,mixed,announced,no,"Lookiero Outfittery Group, in its official res...",Gananzia: Lookiero prepara un ERE pese a recib...,https://gananzia.com/lookiero-prepara-un-ere-p...,lookiero-2026-10,cut,Cited AI among other reasons,Not specified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,BT Group,2023-05-18,NaN,United Kingdom,"Customer service (about 10,000 roles through d...",NaN,10000.0,"The number includes contractors, not only empl...","of 55,000 total cuts",Tier 1,mixed,announced,no,"BT said it would cut up to 55,000 jobs by 2030...",BT Group Newsroom (FY2023 results),https://newsroom.bt.com/results-for-the-full-y...,bt-group-2023-05,cut,Cited AI among other reasons,Customer service and support
316,IBM,2023-05-01,NaN,United States,Back-office; HR,NaN,7800.0,"The number was reported by the press, not stat...",~30% of back-office roles (projected),Tier 2,mixed,announced,no,CEO Arvind Krishna said he could 'easily see 3...,Bloomberg via The National (Krishna interview),https://www.thenationalnews.com/business/techn...,ibm-2023-2025,workforce statement,Spoke about AI and headcount; no cut announced,Operations and back office; Corporate functions
317,Dropbox,2023-04-27,NaN,United States,Research and development; Sales and marketing;...,500.0,NaN,NaN,about 16%,Tier 1,mixed,executed,no,"Dropbox CEO Drew Houston, letter to employees ...",Dropbox Blog: A message from our CEO (Drew Hou...,https://blog.dropbox.com/topics/company/a-mess...,dropbox-2023-04,cut,Cited AI among other reasons,Software engineering and IT; Sales and marketi...
318,Turnitin,2023-04-18,NaN,United States,No roles cut (a forecast that within 18 months...,NaN,NaN,NaN,NaN,Tier 1,expl